# 03 — Database

Loads extracted text from the processor into a SQLite database.

**What this notebook does:**
1. Creates/connects to the SQLite database
2. Creates the `decisions` table if it doesn't exist
3. Reads `.txt` files produced by Notebook 02
4. Cleans text and upserts into the database
5. Prints summary statistics

In [ ]:
import os
import re
import json
import sqlite3
from pathlib import Path
from datetime import datetime

from dotenv import load_dotenv
from tqdm.notebook import tqdm

## Configuration

In [ ]:
load_dotenv()

DOWNLOAD_PATH = Path(os.getenv("DOWNLOAD_PATH", "./downloads"))
DB_PATH = Path(os.getenv("DB_PATH", "./sc_decisions.db"))
MANIFEST_PATH = DOWNLOAD_PATH / "processing_manifest.json"

print(f"Download path: {DOWNLOAD_PATH.resolve()}")
print(f"Database path: {DB_PATH.resolve()}")

## Create Database and Table

In [ ]:
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

cursor.execute("""
    CREATE TABLE IF NOT EXISTS decisions (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        volume TEXT NOT NULL,
        filename TEXT NOT NULL UNIQUE,
        content TEXT,
        page_count INTEGER,
        is_ocr BOOLEAN DEFAULT 0,
        processed_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )
""")

conn.commit()
print("Database ready.")

## Load Processing Manifest

In [ ]:
manifest = {}
if MANIFEST_PATH.exists():
    with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
        manifest = json.load(f)
    print(f"Manifest loaded: {len(manifest)} entries")
else:
    print("No manifest found — will infer metadata from filenames")

## Text Cleaning

In [ ]:
def clean_text(text):
    """Clean extracted text by removing OCR artifacts and excessive whitespace."""
    # Replace multiple blank lines with a single blank line
    text = re.sub(r'\n{3,}', '\n\n', text)

    # Remove lines that are just whitespace
    lines = text.split('\n')
    lines = [line.rstrip() for line in lines]
    text = '\n'.join(lines)

    # Remove common OCR artifacts (isolated single characters on a line)
    text = re.sub(r'\n[|\\/_~`]{1,2}\n', '\n', text)

    # Collapse multiple spaces into one
    text = re.sub(r' {2,}', ' ', text)

    return text.strip()

## Find and Load Text Files

In [ ]:
txt_files = sorted(DOWNLOAD_PATH.glob("*.txt"))
print(f"Found {len(txt_files)} text files to load")

## Upsert into Database

In [ ]:
inserted = 0
updated = 0
errors = 0

for txt_path in tqdm(txt_files, desc="Loading into database"):
    try:
        # Read the text file
        with open(txt_path, "r", encoding="utf-8") as f:
            raw_text = f.read()

        content = clean_text(raw_text)

        # Derive volume name from filename (e.g., Volume_001.txt → Volume_001)
        stem = txt_path.stem
        pdf_filename = stem + ".pdf"

        # Extract volume identifier
        volume_match = re.search(r'Volume[_\s]*(\d+)', stem, re.IGNORECASE)
        volume = volume_match.group(0) if volume_match else stem

        # Get metadata from manifest if available
        meta = manifest.get(pdf_filename, {})
        page_count = meta.get("page_count")
        is_ocr = 1 if meta.get("is_ocr", False) else 0
        processed_at = meta.get("processed_at", datetime.now().isoformat())

        # Check if record already exists
        cursor.execute("SELECT id FROM decisions WHERE filename = ?", (pdf_filename,))
        existing = cursor.fetchone()

        # Upsert
        cursor.execute("""
            INSERT OR REPLACE INTO decisions
                (volume, filename, content, page_count, is_ocr, processed_at)
            VALUES (?, ?, ?, ?, ?, ?)
        """, (volume, pdf_filename, content, page_count, is_ocr, processed_at))

        if existing:
            updated += 1
        else:
            inserted += 1

    except Exception as e:
        print(f"Error processing {txt_path.name}: {e}")
        errors += 1

conn.commit()

## Summary Statistics

In [ ]:
# Load stats
cursor.execute("SELECT COUNT(*) FROM decisions")
total_records = cursor.fetchone()[0]

cursor.execute("SELECT COUNT(*) FROM decisions WHERE is_ocr = 1")
ocr_records = cursor.fetchone()[0]

cursor.execute("SELECT COUNT(*) FROM decisions WHERE is_ocr = 0")
text_records = cursor.fetchone()[0]

cursor.execute("SELECT SUM(page_count) FROM decisions WHERE page_count IS NOT NULL")
total_pages = cursor.fetchone()[0] or 0

cursor.execute("SELECT SUM(LENGTH(content)) FROM decisions WHERE content IS NOT NULL")
total_chars = cursor.fetchone()[0] or 0

summary = (
    f"\n{'='*40}\n"
    f"Database Summary\n"
    f"{'='*40}\n"
    f"This run:\n"
    f"  Inserted:            {inserted}\n"
    f"  Updated:             {updated}\n"
    f"  Errors:              {errors}\n"
    f"\nDatabase totals:\n"
    f"  Total records:       {total_records}\n"
    f"  Text-extracted:      {text_records}\n"
    f"  OCR-processed:       {ocr_records}\n"
    f"  Total pages:         {total_pages:,}\n"
    f"  Total text:          {total_chars:,} chars ({total_chars / 1_000_000:.1f} MB)\n"
    f"  DB file size:        {DB_PATH.stat().st_size / (1024*1024):.1f} MB\n"
    f"{'='*40}"
)
print(summary)

In [ ]:
# Show sample records
cursor.execute("""
    SELECT volume, filename, page_count, is_ocr, LENGTH(content) as text_length
    FROM decisions
    ORDER BY volume
    LIMIT 10
""")

rows = cursor.fetchall()
print(f"{'Volume':<20} {'Filename':<25} {'Pages':>6} {'OCR':>4} {'Text Length':>12}")
print("-" * 70)
for row in rows:
    vol, fname, pages, ocr, tlen = row
    print(f"{vol:<20} {fname:<25} {pages or '-':>6} {'Yes' if ocr else 'No':>4} {tlen or 0:>12,}")

In [ ]:
# Close the connection
conn.close()
print("Database connection closed.")